In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abhishek14398/salary-dataset-simple-linear-regression/Salary_dataset.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/abhishek14398/salary-dataset-simple-linear-regression/Salary_dataset.csv')
df.head()

,Unnamed: 0,YearsExperience,Salary
0,0,1.2,39344.0
1,1,1.4,46206.0
2,2,1.6,37732.0
3,3,2.1,43526.0
4,4,2.3,39892.0


In [4]:
df = df.drop(columns=['Unnamed: 0'])
df.head()

,YearsExperience,Salary
0,1.2,39344.0
1,1.4,46206.0
2,1.6,37732.0
3,2.1,43526.0
4,2.3,39892.0


In [19]:
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, KFold

X = df.drop(columns=['Salary'])
y = df['Salary']

# 1. NORMAL EQUATION (Closed Form)

model = LinearRegression()
model.fit(X,y)

print(f"w (weight/slope)   = {model.coef_[0]:.4f}")
print(f"b (bias/intercept) = {model.intercept_:.4f}")

y_pred = model.predict(X)

print(f"\nPredictions: {np.round(y_pred, 2)}")
print(f"Target: {y}")


mse = mean_squared_error(y, y_pred)
mae = mean_absolute_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"\nMSE  = {mse:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {np.sqrt(mse):.4f}")
print(f"R²   = {r2:.4f}")

scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
mse_scores = -scores
print(scores)

w (weight/slope)   = 9449.9623
b (bias/intercept) = 24848.2040

Predictions: [ 36188.16  38078.15  39968.14  44693.12  46583.12  53198.09  54143.09
  56033.08  56033.08  60758.06  62648.05  63593.05  63593.05  64538.05
  68318.03  72098.02  73988.01  75878.    81547.98  82492.97  90052.94
  92887.93 100447.9  103282.89 108007.87 110842.86 115567.84 116512.84
 123127.81 125017.8 ]
Target: 0      39344.0
1      46206.0
2      37732.0
3      43526.0
4      39892.0
5      56643.0
6      60151.0
7      54446.0
8      64446.0
9      57190.0
10     63219.0
11     55795.0
12     56958.0
13     57082.0
14     61112.0
15     67939.0
16     66030.0
17     83089.0
18     81364.0
19     93941.0
20     91739.0
21     98274.0
22    101303.0
23    113813.0
24    109432.0
25    105583.0
26    116970.0
27    112636.0
28    122392.0
29    121873.0
Name: Salary, dtype: float64

MSE  = 31270951.7223
MAE  = 4644.2013
RMSE = 5592.0436
R²   = 0.9570
[-25905138.76967999 -30974669.99857576 -59780662.11837504
 -

In [6]:
# 2. GRADIENT DESCENT — scikit-learn SGDRegressor

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

sgd = SGDRegressor(
    max_iter=1000,      # max epochs
    learning_rate='constant',
    eta0=0.05, # learning rate
    random_state=42
)
sgd.fit(X_scaled, y)


print(f"w (scaled space) = {sgd.coef_[0]:.4f}")
print(f"b                = {sgd.intercept_[0]:.4f}")
print(f"Total epochs run = {sgd.n_iter_}")


y_pred_sgd = sgd.predict(X_scaled)
print(f"Predictions: {np.round(y_pred_sgd, 2)}")
print(f"MSE = {mean_squared_error(y, y_pred_sgd):.4f}")

w (scaled space) = 26080.9647
b                = 75822.0121
Total epochs run = 16
Predictions: [ 36438.37  38307.85  40177.33  44851.02  46720.5   53263.66  54198.4
  56067.88  56067.88  60741.57  62611.05  63545.78  63545.78  64480.52
  68219.47  71958.43  73827.9   75697.38  81305.81  82240.55  89718.45
  92522.67 100000.57 102804.79 107478.48 110282.69 114956.39 115891.12
 122434.29 124303.77]
MSE = 31385991.9062


In [15]:
# 3. MANUAL GRADIENT DESCENT

def gradient_descent(X, y, lr=0.05, epochs=1000):
    n = len(y)
    w, b = 0.0, 0.0
    x_flat = np.array(X).flatten()
    for epoch in range(epochs):
        y_pred = w * x_flat + b
        error = y - y_pred
        dw = (-2/n) * np.sum(x_flat * error)
        db = (-2/n) * np.sum(error)
        w -= lr * dw
        b -= lr * db
    return w, b
 
w_final, b_final = gradient_descent(X, y, lr=0.05, epochs=1000)
print(f"After 1000 epochs: w = {w_final:.4f}, b = {b_final:.4f}")

from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1,1)).flatten()

w, b = gradient_descent(X_scaled, y_scaled, lr=0.05, epochs=1000)
print(f"w = {w}, b = {b}")


After 1000 epochs: w = nan, b = inf
w = 0.9782416184887595, b = -3.309389799236826e-16


/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/tmp/ipykernel_58/1891154076.py:12: RuntimeWarning: invalid value encountered in scalar subtract
  w -= lr * dw


In [17]:
# 4. TRAIN-TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=1)
print(f"Training flats: {X_train}, rents: {y_train}")
print(f"Test flats:     {X_test}, rents: {y_test}")

Training flats:     YearsExperience
2               1.6
25              9.1
6               3.1
18              6.0
13              4.2
7               3.3
27              9.7
1               1.4
16              5.2
0               1.2
15              5.0
29             10.6
28             10.4
9               3.8
8               3.3
12              4.1
11              4.1
5               3.0, rents: 2      37732.0
25    105583.0
6      60151.0
18     81364.0
13     57082.0
7      54446.0
27    112636.0
1      46206.0
16     66030.0
0      39344.0
15     67939.0
29    121873.0
28    122392.0
9      57190.0
8      64446.0
12     56958.0
11     55795.0
5      56643.0
Name: Salary, dtype: float64
Test flats:         YearsExperience
17              5.4
21              7.2
10              4.0
19              6.1
14              4.6
20              6.9
26              9.6
3               2.1
24              8.8
22              8.0
23              8.3
4               2.3, rents: 17     83089.

In [22]:
# 5. RIDGE vs LASSO 



scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

from sklearn.linear_model import RidgeCV, LassoCV
ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 1000], cv=5)
ridge_cv.fit(X_scaled, y)
print(f"Best alpha = {ridge_cv.alpha_}, w = {ridge_cv.coef_[0]:.4f}")

lasso_cv = LassoCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100, 1000], cv=5)
lasso_cv.fit(X_scaled, y)
print(f"Best alpha = {lasso_cv.alpha_}, w = {lasso_cv.coef_[0]:.4f}")

ridge = Ridge(alpha=ridge_cv.alpha_).fit(X, y)
lasso = Lasso(lasso_cv.alpha_).fit(X, y)
print(f"Plain OLS w = {model.coef_[0]:.4f}")
print(f"Ridge     w = {ridge.coef_[0]:.4f}")
print(f"Lasso     w = {lasso.coef_[0]:.4f}")

Best alpha = 1.0, w = 25516.6282
Best alpha = 1000.0, w = 25367.1824
Plain OLS w = 9449.9623
Ridge     w = 9409.6734
Lasso     w = 9321.5127
